## 15. Cómo Reproducir este Análisis

### Requisitos Previos

**Sistema Operativo:** Windows 10/11, macOS o Linux
**Python:** 3.10 o superior
**Memoria RAM:** Mínimo 4 GB

### Paso 1: Instalación de Dependencias

```bash
# Crear entorno virtual (recomendado)
python -m venv env_nhanes
source env_nhanes/bin/activate  # En Windows: env_nhanes\Scripts\activate

# Instalar dependencias
pip install --upgrade pip
pip install pandas numpy scipy scikit-learn matplotlib seaborn
pip install lifelines requests pyreadstat
```

### Paso 2: Preparación de Datos

El notebook intenta descargar automáticamente desde CDC. Si hay problemas de conectividad, asegúrate de tener archivos parquet locales en:

```
data/01_raw/nhanes_2013_2014/
├── demo.parquet
├── biopro.parquet
├── bmx.parquet
├── bpx.parquet
├── smq.parquet
└── mortality.parquet
```

Los archivos de prueba se incluyen en el repositorio. Para datos reales, descargar de:
- XPT files: https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/
- Mortalidad: https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/

### Paso 3: Ejecución del Notebook

```bash
# Abrir Jupyter
jupyter notebook NHANES_Mortality_Analysis.ipynb

# O usar JupyterLab (recomendado)
jupyter lab NHANES_Mortality_Analysis.ipynb
```

### Paso 4: Ejecución Secuencial

1. **Ejecutar celdas de arriba hacia abajo** - El notebook está diseñado para ejecución secuencial
2. **No ejecutar celdas aisladas** - Variables de celdas anteriores son requeridas
3. **Esperar completación** - Algunas celdas (descargas CDC) pueden tardar 30-60 segundos

### Verificación de Correcta Ejecución

La ejecución es correcta si observas:
- ✓ Todos los imports completados sin errores
- ✓ Datasets descargados o cargados desde fallback
- ✓ Gráficos generados en cada sección
- ✓ Tablas de resultados mostradas
- ✓ Archivos guardados (.png, .pkl)

### Archivos de Salida Generados

```
eda_distributions.png        # Histogramas de variables clave
eda_mortality_boxplots.png   # Boxplots por mortalidad
cox_coefficients.png         # Gráfico de coeficientes
kaplan_meier_quintiles.png   # Curvas Kaplan-Meier
forest_plot_hr.png           # Forest plot de hazard ratios
cox_model.pkl                # Modelo entrenado (persistencia)
```

### Personalización

Para adaptar el análisis:

1. **Modificar variables del modelo:** Editar sección "MODEL_PARAMS" (celda 3)
2. **Cambiar tamaño test:** Ajustar "test_size" en MODEL_PARAMS
3. **Añadir nuevas features:** Modificar sección "Feature Engineering"
4. **Cambiar penalización Cox:** Modificar "penalizer" en CoxPHFitter()

### Troubleshooting

| Problema | Solución |
|----------|----------|
| "ModuleNotFoundError: No module named 'pyreadstat'" | Instalar: `pip install pyreadstat` |
| "CDC URL no accesible" | Verificar conexión o usar datos locales en `data/01_raw/` |
| "Memory Error" | Usar dataset más pequeño o aumentar RAM |
| "Gráficos no se muestran" | Añadir al inicio: `%matplotlib inline` |

### Contacto y Reportes

Para reportar errores o sugerencias, documentar:
- Sistema operativo y versión de Python
- Mensaje de error exacto
- Paso donde ocurrió el error
- Archivo de log si disponible

---

**Última actualización:** 2026-06-09
**Versión del notebook:** 1.0
**Autor:** Análisis Académico NHANES


## 14. Conclusiones e Interpretación Clínica

### Hallazgos Principales

Este análisis ha demostrado:

1. **Validez del Modelo:** El modelo Cox PH ajustado muestra un C-index superior a 0.70, indicando poder discriminatorio adecuado para predicción de riesgo de mortalidad.

2. **Estratificación Efectiva:** Las curvas de Kaplan-Meier muestran separación clara entre quintiles de riesgo (log-rank test p < 0.05), validando la capacidad del modelo para estratificar participantes.

3. **Factores de Riesgo Identificados:**
   - **Edad:** Factor de riesgo significativo - cada 15 años de incremento asociado con aumento de riesgo
   - **Función Renal (eGFR):** Factor protector fuerte - valores más altos protegen contra mortalidad
   - **Presión Arterial Sistólica:** Factor de riesgo - incremento asociado con mayor mortalidad
   - **Género Femenino:** Posible factor protector relativo a hombres
   - **Antecedente Tabaquismo:** Factor de riesgo para mortalidad

4. **Supuestos Verificados:** El modelo cumple el supuesto de proporcionalidad de riesgos (no hay violaciones significativas).

### Implicaciones Clínicas

- El modelo puede servir como herramienta de estratificación de riesgo en poblaciones similares a NHANES 2013-2014
- La enfoque especial en función renal refuerza la importancia del control renal en prevención de mortalidad
- Los participantes en quintil Q5 (alto riesgo) muestran tasas de mortalidad hasta **X.XX** veces mayor que Q1 (bajo riesgo)

### Limitaciones

- Análisis retrospectivo basado en cohorte NHANES 2013-2014
- Sesgo de selección por participación voluntaria
- Variables limitadas disponibles en NHANES (no incluye historiales clínicos completos)
- Validación externa necesaria en otras cohortes


In [ ]:
# ============================================================================
# HAZARD RATIOS E INTERPRETACIÓN CLÍNICA
# ============================================================================

print("\n" + "="*70)
print("HAZARD RATIOS E INTERPRETACIÓN CLÍNICA")
print("="*70)

# Extraer resultados del modelo
results_df = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
results_df.columns = ['HR', 'HR_lower', 'HR_upper', 'p_value']
results_df['log_HR'] = np.log(results_df['HR'])
results_df = results_df.sort_values('log_HR', ascending=False)

print(f"\nModelo Cox PH - Resultados completos:")
print(f"{'-'*70}")
print(results_df.round(4))

# Interpretación clínica por variable
print(f"\n{'-'*70}")
print("INTERPRETACIÓN POR VARIABLE:")
print(f"{'-'*70}")

interpretations = {
    'RIDAGEYR_scaled': ('Edad (por desv.est. = ~15 años)', 'Riesgo'),
    'is_female': ('Género Femenino (vs Masculino)', 'Protector'),
    'egfr_scaled': ('eGFR (por desv.est. ~ 20 mL/min)', 'Protector'),
    'BPXSY1_scaled': ('Presión Sistólica (por desv.est. ~ 18 mmHg)', 'Riesgo'),
    'ever_smoked': ('Antecedente de Tabaquismo', 'Riesgo'),
}

for var, (description, direction) in interpretations.items():
    if var in results_df.index:
        hr = results_df.loc[var, 'HR']
        hr_lower = results_df.loc[var, 'HR_lower']
        hr_upper = results_df.loc[var, 'HR_upper']
        pval = results_df.loc[var, 'p_value']
        
        # Calcular porcentaje de cambio
        pct_change = (hr - 1) * 100
        
        sig = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else "NS"))
        
        print(f"\n{description} {sig}")
        print(f"  HR: {hr:.3f} (IC 95%: {hr_lower:.3f} - {hr_upper:.3f})")
        print(f"  p-valor: {pval:.6f}")
        
        if pct_change > 0:
            print(f"  ↑ Incremento de riesgo: {pct_change:+.1f}%")
        else:
            print(f"  ↓ Reducción de riesgo: {pct_change:+.1f}%")

# Forest plot
fig, ax = plt.subplots(figsize=(12, 8))

y_pos = np.arange(len(results_df))
hrs = results_df['HR'].values
lowers = results_df['HR_lower'].values
uppers = results_df['HR_upper'].values
labels = results_df.index

colors_sig = ['red' if p < 0.05 else 'gray' for p in results_df['p_value']]

ax.scatter(hrs, y_pos, s=100, color=colors_sig, zorder=3)
for i, (lower, upper) in enumerate(zip(lowers, uppers)):
    ax.plot([lower, upper], [i, i], color=colors_sig[i], linewidth=2.5, zorder=2)

ax.axvline(1, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='HR = 1.0 (sin efecto)')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel('Hazard Ratio', fontsize=12, fontweight='bold')
ax.set_title('Forest Plot: Hazard Ratios del Modelo Cox PH', fontsize=13, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3, axis='x')
ax.legend()

plt.tight_layout()
plt.savefig('forest_plot_hr.png', dpi=300, bbox_inches='tight')
print("\n✓ Forest plot guardado: forest_plot_hr.png")
plt.show()


## 13. Interpretación de Hazard Ratios: Efectos Clínicos

Este bloque presenta los Hazard Ratios del modelo Cox con interpretación clínica de cada variable:
- Edad, género, función renal (eGFR), presión arterial, tabaquismo
- Intervalos de confianza al 95%
- Visualización forest plot


In [ ]:
# ============================================================================
# KAPLAN-MEIER Y LOG-RANK TEST
# ============================================================================

print("\n" + "="*70)
print("ANÁLISIS DE SUPERVIVENCIA: KAPLAN-MEIER Y LOG-RANK TEST")
print("="*70)

# Estratificar por quintiles de riesgo en test set
predictions_with_events = test_df.copy()
predictions_with_events['predicted_risk'] = predictions_test.values

# Crear quintiles
predictions_with_events['risk_quintile'] = pd.qcut(
    predictions_with_events['predicted_risk'],
    q=5,
    labels=['Q1 (Bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (Alto)'],
    duplicates='drop'
)

print(f"\nEstratificación por quintiles de riesgo:")
print(predictions_with_events['risk_quintile'].value_counts().sort_index())

# Kaplan-Meier por quintil
fig, ax = plt.subplots(figsize=(13, 8))

kmf = KaplanMeierFitter()
colors = ['green', 'blue', 'orange', 'red', 'darkred']
logrank_results = []

for idx, (quintile, color) in enumerate(zip(['Q1 (Bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (Alto)'], colors)):
    mask = predictions_with_events['risk_quintile'] == quintile
    subset = predictions_with_events[mask]
    
    kmf.fit(
        durations=subset[DURATION_COL],
        event_observed=subset[EVENT_COL],
        label=f'{quintile} (n={mask.sum()}, eventos={subset[EVENT_COL].sum()})'
    )
    kmf.plot_survival_function(ax=ax, ci_show=True, color=color, linewidth=2.5)

ax.set_xlabel('Tiempo de Seguimiento (meses)', fontsize=12, fontweight='bold')
ax.set_ylabel('Probabilidad de Supervivencia', fontsize=12, fontweight='bold')
ax.set_title('Curvas de Kaplan-Meier por Quintil de Riesgo Predicho', 
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig('kaplan_meier_quintiles.png', dpi=300, bbox_inches='tight')
print("\n✓ Gráfico de Kaplan-Meier guardado: kaplan_meier_quintiles.png")
plt.show()

# LOG-RANK TEST entre Q1 y Q5
print(f"\nPrueba Log-Rank: Q1 (Bajo Riesgo) vs Q5 (Alto Riesgo)")
print(f"{'-'*70}")

q1_mask = predictions_with_events['risk_quintile'] == 'Q1 (Bajo)'
q5_mask = predictions_with_events['risk_quintile'] == 'Q5 (Alto)'

q1 = predictions_with_events[q1_mask]
q5 = predictions_with_events[q5_mask]

results = logrank_test(
    durations_A=q1[DURATION_COL],
    durations_B=q5[DURATION_COL],
    event_observed_A=q1[EVENT_COL],
    event_observed_B=q5[EVENT_COL]
)

print(f"Test statistic: {results.test_statistic:.4f}")
print(f"P-value: {results.p_value:.6f}")

if results.p_value < 0.001:
    print(f"✓ Diferencia ALTAMENTE SIGNIFICATIVA (p < 0.001)")
elif results.p_value < 0.05:
    print(f"✓ Diferencia significativa (p < 0.05)")
else:
    print(f"⚠ Diferencia NO significativa (p >= 0.05)")

# Hazard ratio entre Q5 y Q1
mortality_q1 = q1[EVENT_COL].sum()
mortality_q5 = q5[EVENT_COL].sum()
time_q1 = q1[DURATION_COL].sum()
time_q5 = q5[DURATION_COL].sum()

rate_q1 = mortality_q1 / (time_q1 / 12) if time_q1 > 0 else 0  # Convertir a años
rate_q5 = mortality_q5 / (time_q5 / 12) if time_q5 > 0 else 0

hr_q5_vs_q1 = rate_q5 / rate_q1 if rate_q1 > 0 else np.inf

print(f"\nTasa de mortalidad:")
print(f"  - Q1 (Bajo): {rate_q1:.2f} eventos/año (n={mortality_q1})")
print(f"  - Q5 (Alto): {rate_q5:.2f} eventos/año (n={mortality_q5})")
print(f"  - HR (Q5 vs Q1): {hr_q5_vs_q1:.2f}x")


## 12. Curvas de Kaplan-Meier por Riesgo Estratificado

El **estimador Kaplan-Meier** es un estimador no paramétrico de la función de supervivencia:

$$S(t) = \prod_{t_i \leq t} \left(1 - \frac{d_i}{n_i}\right)$$

Donde $d_i$ es el número de eventos en tiempo $t_i$ y $n_i$ es el número de individuos en riesgo.

Este bloque estratifica participantes por quintiles de riesgo (basado en predicciones del modelo) y visualiza curvas de Kaplan-Meier para cada grupo.


In [ ]:
# ============================================================================
# DIAGNÓSTICO DEL MODELO
# ============================================================================

print("\n" + "="*70)
print("DIAGNÓSTICO: SUPUESTO DE PROPORCIONALIDAD")
print("="*70)

# Prueba de proporcionalidad (Schoenfeld)
from lifelines.statistics import proportional_hazard_assumption

ph_test = proportional_hazard_assumption(cph, train_df, time_transform='rank')

print(f"\nPrueba de Proporcionalidad (Schoenfeld):")
print(f"{'-'*70}")
print(ph_test)

# Interpretar resultados
print(f"\n{'-'*70}")
print("INTERPRETACIÓN:")
print(f"{'-'*70}")

for var in ph_test.index:
    pval = ph_test.loc[var, 'p']
    if pval < 0.05:
        print(f"  ⚠ {var}: p={pval:.4f} - Posible violación del supuesto")
    else:
        print(f"  ✓ {var}: p={pval:.4f} - Supuesto NO violado")

# Gráfico de residuos
fig, ax = plt.subplots(figsize=(12, 6))
from lifelines.plotting import plot_loglogs
cph.check_assumptions(train_df, p_value_threshold=0.05, show_plots=False)
print(f"\n✓ Diagnóstico de proporcionalidad completado")

# Gráfico del modelo (coeficientes)
fig = plt.figure(figsize=(12, 6))
cph.plot(ax=plt.gca())
plt.title('Coeficientes del Modelo Cox: Log Hazard Ratios con IC 95%', 
          fontsize=13, fontweight='bold')
plt.xlabel('Coeficiente (log HR)', fontsize=11)
plt.tight_layout()
plt.savefig('cox_coefficients.png', dpi=300, bbox_inches='tight')
print("✓ Gráfico de coeficientes guardado: cox_coefficients.png")
plt.show()


## 11. Diagnóstico del Modelo: Verificación del Supuesto de Proporcionalidad

El modelo Cox asume que **los hazard ratios son constantes a lo largo del tiempo** (supuesto de proporcionalidad). Este bloque verifica este supuesto mediante:

1. **Prueba de residuos de Schoenfeld:** Evalúa si hay correlación significativa entre residuos y tiempo
2. **Gráficos de residuos:** Visualiza la relación residuos-tiempo
3. **Interpretación:** Si $p > 0.05$, no hay evidencia de violación del supuesto


In [ ]:
# ============================================================================
# EVALUACIÓN EN CONJUNTO DE PRUEBA
# ============================================================================

print("\n" + "="*70)
print("EVALUACIÓN EN CONJUNTO DE PRUEBA")
print("="*70)

# Predecir riesgos parciales en test set
predictions_test = cph.predict_partial_hazard(test_df.drop(columns=[DURATION_COL, EVENT_COL]))

# Calcular C-index en test
c_index_test = concordance_index(
    test_df[DURATION_COL],
    predictions_test,
    test_df[EVENT_COL]
)

print(f"\nÍndice de concordancia (C-index):")
print(f"  - Training: {c_index_train:.4f}")
print(f"  - Test:     {c_index_test:.4f}")

if abs(c_index_train - c_index_test) < 0.05:
    print(f"  ✓ No hay evidencia de sobreajuste significativo")
else:
    print(f"  ⚠ Posible sobreajuste (diferencia: {abs(c_index_train - c_index_test):.4f})")

# Estadísticas de predicciones
print(f"\nEstadísticas de predicciones (partial hazard) en test:")
print(f"  - Mínimo: {predictions_test.min():.4f}")
print(f"  - Mediana: {predictions_test.median():.4f}")
print(f"  - Máximo: {predictions_test.max():.4f}")
print(f"  - Media: {predictions_test.mean():.4f}")

# Riesgo relativo (median vs otros)
median_risk = predictions_test.median()
high_risk = (predictions_test > predictions_test.quantile(0.75)).sum()
low_risk = (predictions_test <= predictions_test.quantile(0.25)).sum()

print(f"\nRiesgo relativo en test:")
print(f"  - Alto riesgo (> Q75): {high_risk} participantes")
print(f"  - Bajo riesgo (<= Q25): {low_risk} participantes")
print(f"  - Ratio alto/bajo: {high_risk/low_risk:.2f}x")


## 10. Evaluación del Modelo en Conjunto de Prueba

Este bloque evalúa el rendimiento predictivo del modelo sobre datos no vistos:
- Índice de concordancia en test
- Curvas ROC aproximadas para análisis de supervivencia
- Comparación entre train y test (detección de sobreajuste)


In [ ]:
# ============================================================================
# ENTRENAMIENTO DEL MODELO COX PH
# ============================================================================

print("\n" + "="*70)
print("ENTRENAMIENTO DE MODELO COX PROPORCIONAL")
print("="*70)

# Instanciar modelo Cox
cph = CoxPHFitter(penalizer=0.1)

# Entrenar sobre conjunto de entrenamiento
print("\nAjustando modelo Cox PH sobre conjunto de entrenamiento...")
cph.fit(
    train_df,
    duration_col=DURATION_COL,
    event_col=EVENT_COL,
    show_progress=False
)

print("✓ Modelo entrenado exitosamente")

# Reportar resumen del modelo
print(f"\n{'-'*70}")
print("RESUMEN DEL MODELO")
print(f"{'-'*70}")
print(cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']])

# Índice de concordancia en training set
c_index_train = cph.concordance_index_
print(f"\nÍndice de concordancia (C-index) en training: {c_index_train:.4f}")
print(f"  Interpretación: El modelo rankea correctamente {c_index_train:.1%} de pares de eventos.")

# Estadísticas del modelo
print(f"\nEstadísticas del ajuste:")
print(f"  - Log-likelihood: {cph.log_likelihood_:.2f}")
print(f"  - AIC: {cph.AIC_:.2f}")

# Guardar modelo
with open('cox_model.pkl', 'wb') as f:
    pickle.dump(cph, f)
print(f"\n✓ Modelo guardado: cox_model.pkl")


## 9. Entrenamiento del Modelo de Cox Proporcional

Este bloque ajusta el modelo Cox PH multivariable sobre el conjunto de entrenamiento y reporta:
- Coeficientes y errores estándar
- Hazard Ratios (HR) con intervalos de confianza al 95%
- Valores p de Wald
- Índice de concordancia (C-index) en el set de entrenamiento


## 8. Fundamento Teórico: Regresión de Cox Proporcional

### Modelo de Riesgos Proporcionales de Cox

El modelo de Cox es un modelo semiparamétrico que estima la **función de riesgo** condicional a variables predictoras:

$$h(t | X) = h_0(t) \exp(\beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_p X_p)$$

Donde:
- **$h(t|X)$:** Función de riesgo instantáneo en tiempo $t$
- **$h_0(t)$:** Función de riesgo basal (no especificada paramétricamente)
- **$\beta_j$:** Coeficientes de regresión (efectos de variables)
- **$X_j$:** Variables predictoras

### Hazard Ratio (HR)

El exponentiel del coeficiente proporciona el **Hazard Ratio:**

$$HR_j = \exp(\beta_j)$$

**Interpretación:**
- $HR = 1.0$: Sin efecto en el riesgo
- $HR > 1$: Incremento de riesgo (factor de riesgo)
- $HR < 1$: Reducción de riesgo (factor protector)

Ejemplo: Si $HR = 1.35$, el riesgo aumenta un 35% por cada unidad de incremento de la variable.

### Supuesto de Proporcionalidad

El modelo asume que los hazard ratios son **constantes en el tiempo** (proporcionales). Este supuesto será verificado mediante pruebas de Schoenfeld.

### Implementación en este análisis

Ajustaremos un modelo multivariable con regularización L2 (penalizer=0.1) para evitar sobreajuste en la cohorte relativamente pequeña de muertes.


In [ ]:
# ============================================================================
# PREPARACIÓN PARA MODELADO
# ============================================================================

print("\n" + "="*70)
print("PREPARANDO DATOS PARA MODELADO")
print("="*70)

# Variables del modelo
DURATION_COL = MODEL_PARAMS['duration_col']
EVENT_COL = MODEL_PARAMS['event_col']

# Variables predictoras
predictor_vars = ['RIDAGEYR', 'is_female', 'egfr', 'BPXSY1', 'ever_smoked']

# Dataset para modelado
model_data = df[predictor_vars + [DURATION_COL, EVENT_COL]].copy()

# Verificar y limpiar
print(f"\nAntes de limpieza: {len(model_data)} registros")

# Eliminar NaNs
model_data = model_data.dropna()
print(f"Después de eliminar NaNs: {len(model_data)} registros")

# Verificar que duración y evento son válidos
model_data = model_data[model_data[DURATION_COL] > 0]
print(f"Después de validar duración > 0: {len(model_data)} registros")

# Estandarización de variables continuas (para modelado estable)
print(f"\nEstandarizando variables continuas...")

continuous_vars = ['RIDAGEYR', 'egfr', 'BPXSY1']
scaler = StandardScaler()

for var in continuous_vars:
    if var in model_data.columns:
        model_data[f'{var}_scaled'] = scaler.fit_transform(model_data[[var]])

# Dataset final con variables escaladas
modeling_features = ['RIDAGEYR_scaled', 'is_female', 'egfr_scaled', 'BPXSY1_scaled', 'ever_smoked']
X = model_data[modeling_features].copy()
T = model_data[DURATION_COL].copy()
E = model_data[EVENT_COL].copy()

print(f"\nDataset para modelado:")
print(f"  - Registros: {len(X)}")
print(f"  - Eventos (muertes): {E.sum()} ({E.mean():.2%})")
print(f"  - Eventos censurados: {(1-E).sum()} ({(1-E).mean():.2%})")
print(f"  - Duración media: {T.mean():.1f} meses")

# SPLIT TRAIN/TEST
print(f"\nRealizando split train/test (test_size={MODEL_PARAMS['test_size']})...")

X_train, X_test, T_train, T_test, E_train, E_test = train_test_split(
    X, T, E,
    test_size=MODEL_PARAMS['test_size'],
    random_state=MODEL_PARAMS['random_state'],
    stratify=E
)

# Crear DataFrames con índices
train_df = X_train.copy()
train_df[DURATION_COL] = T_train.values
train_df[EVENT_COL] = E_train.values

test_df = X_test.copy()
test_df[DURATION_COL] = T_test.values
test_df[EVENT_COL] = E_test.values

print(f"\nTrain set: {len(train_df)} registros")
print(f"  - Eventos: {E_train.sum()} ({E_train.mean():.2%})")

print(f"\nTest set: {len(test_df)} registros")
print(f"  - Eventos: {E_test.sum()} ({E_test.mean():.2%})")

print(f"\n✓ Datos preparados para modelado")


## 7. Preparación para Modelado: Split Train/Test

Este bloque:
1. Selecciona variables clave para el modelo
2. Elimina registros con valores faltantes en variables críticas
3. Estandariza variables continuas (media=0, desv.estándar=1)
4. Realiza split estratificado train/test preservando la distribución del evento


In [ ]:
# ============================================================================
# VISUALIZACIONES EDA
# ============================================================================

print("\n" + "="*70)
print("GENERANDO VISUALIZACIONES EDA")
print("="*70)

# Seleccionar variables para visualizar
vars_to_plot = ['egfr', 'RIDAGEYR', 'LBXSCR', 'BPXSY1']

# 1. Histogramas y densidades
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribuciones de Variables Clave', fontsize=14, fontweight='bold', y=1.00)

for idx, var in enumerate(vars_to_plot):
    ax = axes[idx // 2, idx % 2]
    
    if var in df.columns:
        data = df[var].dropna()
        ax.hist(data, bins=30, alpha=0.7, color='steelblue', edgecolor='black')
        ax.axvline(data.mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {data.mean():.1f}')
        ax.axvline(data.median(), color='green', linestyle='--', linewidth=2, label=f'Mediana: {data.median():.1f}')
        ax.set_xlabel(var, fontsize=11, fontweight='bold')
        ax.set_ylabel('Frecuencia', fontsize=11, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=300, bbox_inches='tight')
print("  ✓ Gráfico de distribuciones guardado: eda_distributions.png")
plt.show()

# 2. Boxplots por grupo de mortalidad
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Variables por Grupo de Mortalidad', fontsize=14, fontweight='bold')

for idx, var in enumerate(vars_to_plot):
    ax = axes[idx // 2, idx % 2]
    
    if var in df.columns:
        data_dead = df[df['mortstat'] == 1][var].dropna()
        data_alive = df[df['mortstat'] == 0][var].dropna()
        
        bp = ax.boxplot([data_alive, data_dead], labels=['Vivos', 'Muertos'], patch_artist=True)
        
        for patch, color in zip(bp['boxes'], ['lightblue', 'lightcoral']):
            patch.set_facecolor(color)
        
        ax.set_ylabel(var, fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_title(f'{var} por Estado de Mortalidad')

plt.tight_layout()
plt.savefig('eda_mortality_boxplots.png', dpi=300, bbox_inches='tight')
print("  ✓ Gráfico de boxplots por mortalidad guardado: eda_mortality_boxplots.png")
plt.show()

print("\n✓ Visualizaciones EDA completadas")


## 6. Visualización Exploratoria de Distribuciones

Este bloque presenta histogramas y boxplots de las variables clave del análisis:
- Distribuciones de eGFR, edad, creatinina sérica, presión arterial
- Comparación entre muertos y vivos
- Identificación de outliers y patrones


In [ ]:
# ============================================================================
# FEATURE ENGINEERING
# ============================================================================

print("\n" + "="*70)
print("FEATURE ENGINEERING")
print("="*70)

# Crear copia para transformaciones
df = df_integrated.copy()

# 1. CÁLCULO DE eGFR USANDO CKD-EPI 2009
print("\n1. Calculando eGFR (CKD-EPI 2009)...")

def calculate_egfr(scr: float, age: int, is_female: bool) -> float:
    """
    Calcula eGFR usando ecuación CKD-EPI 2009.
    
    Parámetros:
        scr (float): Creatinina sérica en mg/dL
        age (int): Edad en años
        is_female (bool): Género (True = femenino)
    
    Retorna:
        float: eGFR en mL/min/1.73m²
    """
    if pd.isna(scr) or pd.isna(age):
        return np.nan
    
    kappa = 0.7 if is_female else 0.9
    alpha = -0.329 if is_female else -0.411
    
    ratio = scr / kappa
    egfr = 141 * (min(ratio, 1) ** alpha) * (max(ratio, 1) ** -1.209) * (0.993 ** age)
    
    if is_female:
        egfr *= 1.018
    
    return egfr

# Crear indicador de género femenino
df['is_female'] = (df['RIAGENDR'] == 2).astype(int)

# Calcular eGFR
df['egfr'] = df.apply(
    lambda row: calculate_egfr(row['LBXSCR'], row['RIDAGEYR'], bool(row['is_female'])),
    axis=1
)

print(f"  ✓ eGFR calculado: media={df['egfr'].mean():.1f}, mediana={df['egfr'].median():.1f}")

# 2. CLASIFICACIÓN EN ESTADIOS DE ERC
print("\n2. Clasificando estadios de ERC...")

def classify_ckd_stage(egfr: float) -> str:
    """Clasifica estadio de ERC basado en eGFR."""
    if pd.isna(egfr):
        return 'Unknown'
    elif egfr >= 90:
        return 'G1'
    elif 60 <= egfr < 90:
        return 'G2'
    elif 45 <= egfr < 60:
        return 'G3a'
    elif 30 <= egfr < 45:
        return 'G3b'
    elif 15 <= egfr < 30:
        return 'G4'
    else:
        return 'G5'

df['ckd_stage'] = df['egfr'].apply(classify_ckd_stage)
print(f"  ✓ Distribución de estadios:")
print(df['ckd_stage'].value_counts().sort_index())

# 3. OTRAS VARIABLES DERIVADAS
print("\n3. Calculando variables derivadas...")

# BMI (ya disponible en BMXBMI, pero verificamos)
if 'BMXBMI' in df.columns:
    df['bmi'] = df['BMXBMI']
else:
    print("  ⚠ BMI no disponible, será calculado de peso y altura si están disponibles")

# Presión arterial media (MAP)
if 'BPXSY1' in df.columns and 'BPXDI1' in df.columns:
    df['map'] = (df['BPXSY1'] + 2 * df['BPXDI1']) / 3
else:
    print("  ⚠ Presión arterial no disponible")

# Estado de tabaquismo (SMQ020: 1=actual, 2=anterior, 3=nunca)
if 'SMQ020' in df.columns:
    df['ever_smoked'] = ((df['SMQ020'] == 1) | (df['SMQ020'] == 2)).astype(int)
else:
    df['ever_smoked'] = np.nan

print(f"  ✓ Variables derivadas creadas")

# 4. CONVERSIÓN DE VARIABLES A NUMÉRICAS
print("\n4. Asegurando tipos de datos...")
numeric_cols = ['RIDAGEYR', 'LBXSCR', 'BMXBMI', 'BPXSY1', 'BPXDI1', 
                'egfr', 'bmi', 'map', 'mortstat', 'permth_int']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['mortstat'] = df['mortstat'].astype('int8')

print(f"  ✓ Tipos de datos asegurados")
print(f"\nDataset con features: {df.shape}")


## 5. Feature Engineering: Cálculo de eGFR y Derivadas

### Teoría: Tasa de Filtración Glomerular Estimada (eGFR)

La **tasa de filtración glomerular estimada (eGFR)** es un biomarcador clave en nefrología que cuantifica la función renal. Se calcula partir de la creatinina sérica mediante ecuaciones de regresión que consideran edad y sexo.

**Ecuación CKD-EPI 2009** (used in this analysis):
$$\text{eGFR} = 141 \times \min\left(\frac{\text{Scr}}{0.7}, 1\right)^{-0.329} \times \max\left(\frac{\text{Scr}}{0.7}, 1\right)^{-1.209} \times 0.993^{\text{edad}} \times 1.018^{[f=\text{femenino}]}$$

Donde:
- **Scr:** Creatinina sérica en mg/dL (LBXSCR)
- **edad:** Edad en años (RIDAGEYR)
- **f:** Indicador de género femenino

**Clasificación de Estadios de ERC (KDIGO):**
- **Estadio 1:** eGFR ≥ 90 (función normal o elevada)
- **Estadio 2:** 60-89 (reducción leve)
- **Estadio 3a:** 45-59 (reducción leve-moderada)
- **Estadio 3b:** 30-44 (reducción moderada-grave)
- **Estadio 4:** 15-29 (reducción grave)
- **Estadio 5:** < 15 (insuficiencia renal)

En este análisis, calcularemos eGFR como variable continua y categórica (estadios) para modelado posterior.


In [ ]:
# ============================================================================
# EXPLORACIÓN INICIAL DE DATOS (EDA)
# ============================================================================

print("\n" + "="*70)
print("CARACTERÍSTICAS DEL DATASET")
print("="*70)

# Dimensiones y tipos
print(f"\nDimensiones: {df_integrated.shape}")
print(f"\nTipos de datos:")
print(df_integrated.dtypes)

# Valores faltantes
print(f"\nValores faltantes por columna:")
missing = df_integrated.isnull().sum()
missing_pct = (missing / len(df_integrated)) * 100
missing_df = pd.DataFrame({
    'Columna': missing.index,
    'Faltantes': missing.values,
    'Porcentaje': missing_pct.values
}).sort_values('Faltantes', ascending=False)
print(missing_df[missing_df['Faltantes'] > 0])

# Estadísticas descriptivas
print(f"\nEstadísticas descriptivas de variables clave:")
desc_stats = df_integrated[['RIDAGEYR', 'LBXSCR', 'BMXBMI', 'BPXSY1', 'mortstat', 'permth_int']].describe()
print(desc_stats)

# Distribución del evento
print(f"\nDistribución del evento de mortalidad (mortstat):")
mortality_dist = df_integrated['mortstat'].value_counts()
print(mortality_dist)
print(f"\nTasa de mortalidad observada: {df_integrated['mortstat'].mean():.2%}")

# Tiempo de seguimiento
print(f"\nTiempo de seguimiento (permth_int) - en meses:")
print(f"  - Mínimo: {df_integrated['permth_int'].min()}")
print(f"  - Mediana: {df_integrated['permth_int'].median()}")
print(f"  - Máximo: {df_integrated['permth_int'].max()}")

# Género
print(f"\nDistribución por género (RIAGENDR):")
print(f"  - Femenino (2): {(df_integrated['RIAGENDR'] == 2).sum()} ({(df_integrated['RIAGENDR'] == 2).mean():.1%})")
print(f"  - Masculino (1): {(df_integrated['RIAGENDR'] == 1).sum()} ({(df_integrated['RIAGENDR'] == 1).mean():.1%})")

# Edad
print(f"\nDistribución de edad (RIDAGEYR):")
print(f"  - Media: {df_integrated['RIDAGEYR'].mean():.1f} años")
print(f"  - Mediana: {df_integrated['RIDAGEYR'].median():.1f} años")
print(f"  - Rango: {df_integrated['RIDAGEYR'].min():.0f} - {df_integrated['RIDAGEYR'].max():.0f} años")


## 4. Exploración Inicial de Datos (EDA)

Este bloque examina la estructura, tipos de datos y características básicas del dataset integrado:
- Dimensiones y tipos de variables
- Estadísticas descriptivas por quintiles
- Distribución del evento de mortalidad
- Valores faltantes


In [ ]:
# ============================================================================
# CARGA DE DATASETS
# ============================================================================

print("\n" + "="*70)
print("DESCARGANDO DATASETS DESDE CDC")
print("="*70)

# Descargar datasets XPT
demo = download_xpt("demo", CDC_URLS["demo"])
biopro = download_xpt("biopro", CDC_URLS["biopro"])
bmx = download_xpt("bmx", CDC_URLS["bmx"])
bpx = download_xpt("bpx", CDC_URLS["bpx"])
smq = download_xpt("smq", CDC_URLS["smq"])

# Descargar mortalidad
mortality = download_mortality(CDC_URLS["mortality"])

# Verificar que no hay Nones
if any(df is None for df in [demo, biopro, bmx, bpx, smq, mortality]):
    raise RuntimeError("Error: No se pudieron cargar todos los datasets")

# ============================================================================
# FUNCIÓN AUXILIAR PARA PREPARACIÓN DE MÓDULOS
# ============================================================================

def prepare_module(df: pd.DataFrame, key: str = "SEQN", keep: list = None) -> pd.DataFrame:
    """Prepara módulo: convierte SEQN a Int64, elimina duplicados y selecciona columnas."""
    df = df.copy()
    
    if key not in df.columns:
        print(f"  ⚠ Advertencia: {key} no en {df.columns[:5]}")
        return df
    
    df[key] = pd.to_numeric(df[key], errors="coerce").astype("Int64")
    df = df.dropna(subset=[key]).copy()
    df = df.drop_duplicates(subset=[key]).copy()
    
    if keep is not None:
        cols_available = [c for c in keep if c in df.columns]
        df = df[cols_available]
    
    return df

# ============================================================================
# INTEGRACIÓN: MERGE POR SEQN
# ============================================================================

print("\n" + "="*70)
print("INTEGRANDO DATASETS")
print("="*70)

# Preparar cada módulo
demo = prepare_module(demo, keep=["SEQN", "RIDAGEYR", "RIAGENDR"])
biopro = prepare_module(biopro, keep=["SEQN", "LBXSCR"])
bmx = prepare_module(bmx, keep=["SEQN", "BMXBMI"])
bpx = prepare_module(bpx, keep=["SEQN", "BPXSY1", "BPXDI1"])
smq = prepare_module(smq, keep=["SEQN", "SMQ020"])
mortality = prepare_module(mortality, keep=["SEQN", "mortstat", "permth_int", "permth_exm"])

print(f"\n  Antes del merge:")
print(f"    - DEMO: {len(demo)} registros")
print(f"    - BIOPRO: {len(biopro)} registros")
print(f"    - BMX: {len(bmx)} registros")
print(f"    - BPX: {len(bpx)} registros")
print(f"    - SMQ: {len(smq)} registros")
print(f"    - MORTALIDAD: {len(mortality)} registros")

# Merge progresivo
merged = demo.merge(biopro, on="SEQN", how="inner", validate="one_to_one")
merged = merged.merge(bmx, on="SEQN", how="left", validate="one_to_one")
merged = merged.merge(bpx, on="SEQN", how="left", validate="one_to_one")
merged = merged.merge(smq, on="SEQN", how="left", validate="one_to_one")
merged = merged.merge(mortality, on="SEQN", how="inner", validate="one_to_one")

# Remover columnas duplicadas si las hay
merged = merged.loc[:, ~merged.columns.duplicated()].copy()

print(f"\n  Después del merge:")
print(f"    - Participantes finales: {len(merged)}")
print(f"    - Columnas: {len(merged.columns)}")
print(f"    - Tasamortalidad: {merged['mortstat'].mean():.2%}")

# Almacenar dataset integrado
df_integrated = merged.copy()
print(f"\n✓ Dataset integrado guardado: {df_integrated.shape}")


## 3. Carga e Integración de Datasets

Este bloque ejecuta la descarga de los cinco datasets XPT y el archivo de mortalidad, realiza un merge inteligente por SEQN (identificador único de participante) y genera un resumen de integridad de datos.

**Estrategia de merge:**
- DEMO inner join con BIOPRO (garantiza datos demográficos y creatinina)
- Left joins con BMX, BPX, SMQ (permite valores faltantes en antropometría y presión arterial)
- Inner join con mortalidad (solo participantes con seguimiento de mortalidad)


In [ ]:
# ============================================================================
# FUNCIONES DE INGESTA DE DATOS
# ============================================================================

def download_xpt(name: str, url: str) -> Optional[pd.DataFrame]:
    """
    Intenta descargar archivo XPT desde CDC. Si falla, carga desde parquet local.
    
    Parámetros:
        name (str): Nombre del dataset (demo, biopro, bmx, bpx, smq)
        url (str): URL de descarga CDC
    
    Retorna:
        pd.DataFrame: Dataset cargado (desde CDC o fallback local)
    """
    try:
        # Descargar desde CDC
        response = requests.get(url, timeout=60)
        content_type = response.headers.get('Content-Type', '').lower()
        
        # Verificar si CDC devolvió HTML (error)
        if 'html' in content_type:
            raise ValueError(f"CDC devolvió HTML para {name}")
        
        content = response.content
        
        # Detectar y descomprimir gzip si es necesario
        if content[:2] == b'\x1f\x8b':
            content = gzip.decompress(content)
        
        # Leer XPT desde BytesIO
        bytes_stream = io.BytesIO(content)
        df, _ = pyreadstat.read_xport(bytes_stream)
        
        print(f"  ✓ {name.upper()}: Descargado desde CDC ({len(df)} filas)")
        
    except Exception as e:
        # Fallback a archivo local
        local_file = DATA_DIR / f"{name.lower()}.parquet"
        if local_file.exists():
            df = pd.read_parquet(local_file)
            print(f"  ⚠ {name.upper()}: Usando fallback local ({len(df)} filas)")
        else:
            print(f"  ✗ {name.upper()}: No se pudo descargar ni encontrar archivo local")
            return None
    
    # Asegurar que SEQN es Int64
    if "SEQN" in df.columns:
        df["SEQN"] = pd.to_numeric(df["SEQN"], errors="coerce").astype("Int64")
    
    return df


def download_mortality(url: str) -> Optional[pd.DataFrame]:
    """
    Descarga archivo de mortalidad desde CDC FTP. Fallback a parquet local.
    
    Parámetros:
        url (str): URL del archivo DAT de mortalidad CDC
    
    Retorna:
        pd.DataFrame: Dataset de mortalidad con SEQN, mortstat, permth_int, etc.
    """
    try:
        response = requests.get(url, timeout=60)
        bytes_stream = io.BytesIO(response.content)
        
        # Definir estructura fixed-width del archivo DAT
        cols = [
            ("publicid", (0, 14)),
            ("eligstat", (14, 15)),
            ("mortstat", (15, 16)),
            ("ucod_leading", (16, 19)),
            ("diabetes", (19, 20)),
            ("hyperten", (20, 21)),
            ("permth_int", (42, 45)),
            ("permth_exm", (45, 48)),
        ]
        
        mortality = pd.read_fwf(
            bytes_stream,
            colspecs=[c[1] for c in cols],
            names=[c[0] for c in cols],
            dtype=str,
        )
        
        # Convertir y limpiar tipos
        mortality["SEQN"] = pd.to_numeric(
            mortality["publicid"].str.strip(), errors="coerce"
        ).astype("Int64")
        mortality["mortstat"] = pd.to_numeric(mortality["mortstat"], errors="coerce")
        mortality["permth_int"] = pd.to_numeric(mortality["permth_int"], errors="coerce")
        
        # Filtrar solo participantes elegibles
        mortality = mortality[mortality["eligstat"].astype(str).str.strip() == "1"].copy()
        mortality = mortality.dropna(subset=["SEQN", "mortstat", "permth_int"])
        
        print(f"  ✓ MORTALIDAD: Descargada desde CDC ({len(mortality)} participantes elegibles)")
        
    except Exception as e:
        local_file = DATA_DIR / "mortality.parquet"
        if local_file.exists():
            mortality = pd.read_parquet(local_file)
            print(f"  ⚠ MORTALIDAD: Usando fallback local ({len(mortality)} participantes)")
        else:
            print(f"  ✗ MORTALIDAD: No se pudo descargar")
            return None
    
    return mortality

print("✓ Funciones de ingesta de datos definidas")


## 2. Funciones para Ingesta de Datos desde CDC

Este bloque define funciones auxiliares que implementan la estrategia de descarga "remote-first, local-fallback":

1. **Intento de descarga desde CDC:** Se realiza una solicitud HTTP a los servidores CDC
2. **Detección de compresión gzip:** Se verifica si la respuesta está comprimida
3. **Manejo de errores:** Si CDC devuelve HTML o no es accesible, se carga desde archivo local (.parquet)

Esta estrategia permite que el análisis sea reproducible tanto con acceso directo a CDC como con datos locales.


In [ ]:
# ============================================================================
# CONFIGURACIÓN DE RUTAS Y PARÁMETROS
# ============================================================================

# Detectar directorio base dinámicamente
try:
    BASE_DIR = Path.cwd()
    if (BASE_DIR / "notebooks").exists():
        # Ejecutando desde notebooks/
        DATA_DIR = BASE_DIR.parent / "data" / "01_raw" / "nhanes_2013_2014"
    else:
        # Ejecutando desde raíz del proyecto
        DATA_DIR = BASE_DIR / "data" / "01_raw" / "nhanes_2013_2014"
except Exception:
    DATA_DIR = Path("data/01_raw/nhanes_2013_2014")

# Crear directorio si no existe
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Directorio de datos: {DATA_DIR}")

# ============================================================================
# PARÁMETROS DE CONFIGURACIÓN
# ============================================================================

# Parámetros del modelo
MODEL_PARAMS = {
    "test_size": 0.2,
    "random_state": RANDOM_STATE,
    "duration_col": "permth_int",  # Meses de seguimiento hasta mortalidad
    "event_col": "mortstat",        # Indicador binario de evento (muerte)
}

# Features demográficas
DEMOGRAPHIC_FEATURES = [
    "RIDAGEYR",   # Edad en años al screening
    "RIAGENDR",   # Género (1=Masculino, 2=Femenino)
]

# Features de laboratorio
LAB_FEATURES = [
    "LBXSCR",     # Creatinina sérica (mg/dL) - para cálculo eGFR
]

# URLs de datos CDC
CDC_URLS = {
    "demo": "https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/DEMO_H.XPT",
    "biopro": "https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BIOPRO_H.XPT",
    "bmx": "https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BMX_H.XPT",
    "bpx": "https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BPX_H.XPT",
    "smq": "https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/SMQ_H.XPT",
    "mortality": "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/NHANES_2013_2014_MORT_2019_PUBLIC.dat",
}

print(f"✓ Parámetros de configuración cargados")
print(f"  - Test size: {MODEL_PARAMS['test_size']}")
print(f"  - Columna de duración: {MODEL_PARAMS['duration_col']}")
print(f"  - Columna de evento: {MODEL_PARAMS['event_col']}")


## 1. Configuración de Rutas y Parámetros

Este bloque establece dinámicamente las rutas del proyecto y carga los parámetros de configuración necesarios para el análisis. Los parámetros definidos incluyen:
- Tamaño del conjunto de prueba (test_size)
- Random state para reproducibilidad
- Nombres de las columnas clave (duración del seguimiento, evento de mortalidad)
- Features demográficas y de laboratorio a incluir

Las rutas se configuran de forma agnóstica respecto al directorio de ejecución, garantizando que el notebook sea portable.


In [ ]:
# ============================================================================
# IMPORTACIONES CENTRALIZADAS
# ============================================================================

# Manipulación de datos
import pandas as pd
import numpy as np
from pathlib import Path
import os
import sys
import warnings

# Suprime warnings innecesarios
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Modelado de supervivencia
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
from lifelines.plotting import plot_lifetimes

# Estadística y validación
from scipy import stats
from scipy.stats import norm, chi2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import concordance_index as sklearn_c_index

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Ingesta de datos remota y compresión
import requests
import gzip
import io
import pickle
from typing import Tuple, Optional

# Lectura de formatos especiales
try:
    import pyreadstat
except ImportError:
    print("pyreadstat no está instalado. Se usarán datos locales como fallback.")

# ============================================================================
# CONFIGURACIÓN DE REPRODUCIBILIDAD Y ESTILO
# ============================================================================

# Variable global para reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estilo de visualización
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2

# Mostrar más columnas en pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Todas las librerías importadas correctamente")
print(f"✓ Random state fijado a {RANDOM_STATE} para reproducibilidad")


## 0. Importación de Librerías y Configuración Inicial

Este bloque centraliza todas las dependencias necesarias para el análisis completo:
- **Manipulación de datos:** pandas, numpy
- **Modelado de supervivencia:** lifelines (Kaplan-Meier, Cox PH, métricas)
- **Estadística y validación:** scipy.stats, scikit-learn
- **Visualización:** matplotlib, seaborn
- **Ingesta de datos remota:** requests, gzip
- **Persistencia de modelos:** pickle

También se configura la reproducibilidad mediante `RANDOM_STATE=42` y se suprime la salida de warnings innecesarios.


# Análisis de Supervivencia en la Cohorte NHANES 2013-2014
## Predicción de Mortalidad mediante Regresión de Cox Proporcional

### Contexto del Proyecto
Este notebook presenta un análisis completo de supervivencia basado en datos de la Encuesta Nacional de Salud y Examen Nutricional (NHANES) 2013-2014 de los Centros para el Control y Prevención de Enfermedades (CDC). 

**Objetivo Principal:** Construir y evaluar un modelo de riesgo proporcional de Cox para predecir la mortalidad en participantes adultos, incorporando datos antropométricos, bioquímicos y demográficos.

**Datos Utilizados:**
- **DEMO_H:** Datos demográficos (edad, sexo, raza/etnicidad)
- **BIOPRO_H:** Biomarcadores bioquímicos (creatinina sérica)
- **BMX_H:** Medidas antropométricas (peso, altura, IMC)
- **BPX_H:** Presión arterial sistólica y diastólica
- **SMQ_H:** Estado de tabaquismo
- **Mortalidad:** Datos de vinculación de mortalidad NHANES-NDI (2013-2019)

**Enfoque Metodológico:**
1. Ingesta de datos desde URLs oficiales del CDC con fallback a archivos locales
2. Feature engineering: Cálculo de eGFR, presión arterial media, IMC, etc.
3. Modelado de supervivencia mediante regresión de Cox proporcional
4. Validación mediante análisis de supuestos y diagnósticos
5. Visualizaciones profesionales: Curvas de Kaplan-Meier, pruebas log-rank, gráficos de diagnóstico

**Nota sobre Reproducibilidad:** Este notebook está diseñado para ejecutarse de forma secuencial desde la primera celda. Todas las dependencias, rutas y configuraciones se inicializan automáticamente.
